In [1]:
import pandas as pd
import numpy as np

In [4]:
# Future routes
routes_future = pd.read_csv('./output/future/routes.txt')
routes_future

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
0,L-PLR1-CLFD-WSMD,SLR,L-PLR1,L Carlingford - Westmead,Sydney Light Rail Network,0,EE343F,FFFFFF,0
1,L-PLR1-WSMD-CLFD,SLR,L-PLR1,L Westmead - Carlingford,Sydney Light Rail Network,0,EE343F,FFFFFF,0
2,L-PLR2-OLYP-WSUN,SLR,L-PLR2,L Olympic Park - Western Sydney University Cam...,Sydney Light Rail Network,0,EE343F,FFFFFF,0
3,L-PLR2-WSUN-OLYP,SLR,L-PLR2,"L Western Sydney University Campus, Parramatta...",Sydney Light Rail Network,0,EE343F,FFFFFF,0
4,M-1-LVPL-TLWG,SMNW,M-1,M Liverpool - Tallawong,Sydney Metro Network,1,008C96,FFFFFF,0
5,M-1-TLWG-LVPL,SMNW,M-1,M Tallawong - Liverpool,Sydney Metro Network,1,008C96,FFFFFF,0
6,M-2-BKTN-EPNG,SMNW,M-2,M Bankstown - Epping,Sydney Metro Network,1,008C96,FFFFFF,0
7,M-2-EPNG-BKTN,SMNW,M-2,M Epping - Bankstown,Sydney Metro Network,1,008C96,FFFFFF,0
8,M-3a-MBRA-WSIA,SMNW,M-3a,M Maroubra - Western Sydney Aerotropolis,Sydney Metro Network,1,008C96,FFFFFF,0
9,M-3a-WSIA-MBRA,SMNW,M-3a,M Western Sydney Aerotropolis - Maroubra,Sydney Metro Network,1,008C96,FFFFFF,0


In [8]:
routes_future['route_id'].unique()

array(['L-PLR1-CLFD-WSMD', 'L-PLR1-WSMD-CLFD', 'L-PLR2-OLYP-WSUN',
       'L-PLR2-WSUN-OLYP', 'M-1-LVPL-TLWG', 'M-1-TLWG-LVPL',
       'M-2-BKTN-EPNG', 'M-2-EPNG-BKTN', 'M-3a-MBRA-WSIA',
       'M-3a-WSIA-MBRA', 'M-3b-MBRA-PRMT', 'M-3b-PRMT-MBRA',
       'M-4a-KGRH-TLWG', 'M-4a-TLWG-KGRH', 'M-4b-KGRH-NWST',
       'M-4b-NWST-KGRH', 'M-5-EPNG-WSIA', 'M-5-WSIA-EPNG',
       'M-6-MCTR-TLWG', 'M-6-TLWG-MCTR', 'T2-CTYC-WSAR', 'T2-WSAR-CTYC'],
      dtype=object)

In [9]:
routesToRemove = ['M-4a-KGRH-TLWG', 'M-4a-TLWG-KGRH', 'M-4b-KGRH-NWST',
       'M-4b-NWST-KGRH', 'M-5-EPNG-WSIA', 'M-5-WSIA-EPNG',
       'M-6-MCTR-TLWG', 'M-6-TLWG-MCTR']

In [18]:
# Read combined gtfs

agency = pd.read_csv('./output/combined/agency.txt')
calendardates = pd.read_csv('./output/combined/calendar_dates.txt')
calendar = pd.read_csv('./output/combined/calendar.txt')
routes = pd.read_csv('./output/combined/routes.txt')
shapes = pd.read_csv('./output/combined/shapes.txt')
trips = pd.read_csv('./output/combined/trips.txt')
stoptimes = pd.read_csv('./output/combined/stop_times.txt')
stops = pd.read_csv('./output/combined/stops.txt')

In [19]:
# Remove M4, M5 and M6

print(len(routes['route_id']))
routes = routes.loc[routes['route_id'].isin(routesToRemove)==False]
print(len(routes['route_id']))

print(len(trips))
trips['route_id'] = trips['route_id'].astype(str)
routes['route_id'] = routes['route_id'].astype(str)
trips = trips.loc[trips['route_id'].isin(routes['route_id'])]
print(len(trips))

print(len(stoptimes))
stoptimes['trip_id'] = stoptimes['trip_id'].astype(str)
trips['trip_id'] = trips['trip_id'].astype(str)
stoptimes = stoptimes.loc[stoptimes['trip_id'].isin(trips['trip_id'])]
print(len(stoptimes))

print(len(stops))
stops['stop_id'] = stops['stop_id'].astype(str)
stoptimes['stop_id'] = stoptimes['stop_id'].astype(str)
stops = stops.loc[(stops['stop_id'].isin(stoptimes['stop_id'])) | (stops['stop_id'].str.startswith('PST'))]
print(len(stops))

print(len(shapes))
shapes['shape_id'] = shapes['shape_id'].astype(str)
trips['shape_id'] = trips['shape_id'].astype(str)
shapes = shapes.loc[shapes['shape_id'].isin(trips['shape_id'])]
print(len(shapes))

print(len(calendar))
calendar['service_id'] = calendar['service_id'].astype(str)
trips['service_id'] = trips['service_id'].astype(str)
calendar = calendar.loc[calendar['service_id'].isin(trips['service_id'])]
print(len(calendar))

print(len(calendardates))
calendardates['service_id'] = calendardates['service_id'].astype(str)
trips['service_id'] = trips['service_id'].astype(str)
calendardates = calendardates.loc[calendardates['service_id'].isin(trips['service_id'])]
print(len(calendardates))


6633
6625
150516
149388
4389825
4374597
45073
45002
8206517
8205375
2160
2160
25779
25779


In [20]:
# Edit data types (to avoid validation errors)

stoptimes['pickup_type'] = stoptimes['pickup_type'].apply(lambda x: str(int(x)) if x>=0 else '')
stoptimes['drop_off_type'] = stoptimes['drop_off_type'].apply(lambda x: str(int(x)) if x>=0 else '')
trips['wheelchair_accessible'] = trips['wheelchair_accessible'].apply(lambda x: str(int(x)) if x>=0 else '')
trips['bikes_allowed'] = trips['bikes_allowed'].apply(lambda x: str(int(x)) if x>=0 else '')
stops['location_type'] = stops['location_type'].apply(lambda x: str(int(x)) if x>=0 else '')



In [21]:
routes.to_csv('./output/Parramatta/routes.txt', index=False)
shapes.to_csv('./output/Parramatta/shapes.txt', index=False)
trips.to_csv('./output/Parramatta/trips.txt', index=False)
stops.to_csv('./output/Parramatta/stops.txt', index=False)
stoptimes.to_csv('./output/Parramatta/stop_times.txt', index=False)
calendar.to_csv('./output/Parramatta/calendar.txt', index=False)
calendardates.to_csv('./output/Parramatta/calendar_dates.txt', index=False)
agency.to_csv('./output/Parramatta/agency.txt', index=False)